# QLoRA Fine-Tuning Llama-2 for Sentiment Generation — Reference Notebook

> **Reference notebook.** See [`fine_tuning.md`](./fine_tuning.md) for the LoRA/QLoRA math and PEFT
> concepts behind every config value used here, and [`README.md`](./README.md) for the fuller
> walkthrough this notebook distills.

**Methods covered:**
- Loading a base causal LM in 4-bit (`BitsAndBytesConfig`, NF4) via `AutoModelForCausalLM`
- Attaching trainable LoRA adapters (`LoraConfig`) on top of the frozen, quantized base model
- Supervised fine-tuning on pre-formatted instruction-tagged text with `SFTTrainer`/`SFTConfig`
- Running inference through the adapter-attached model
- Merging LoRA adapters into the base model (`PeftModel.merge_and_unload()`) for standalone deployment

**Use this as a reference when:** you need copy-paste-ready code for QLoRA fine-tuning a decoder-only
Hugging Face model and deploying it either with adapters attached or merged into a standalone model.

**Don't use this as a reference for:** encoder-decoder (seq2seq) fine-tuning — see
[module 09](../09-legal-assistant-llm-finetuning/README.md) instead — or evaluation methodology, which
this notebook doesn't cover.

In [ ]:
import torch
import gc

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Pre-formatted CSV: each row is already a single Llama-2 chat-template string
# "<s>[INST] {review} [/INST] {Positive|Negative} </s>" -- no separate preprocessing needed.
dataset = load_dataset("csv", data_files="dataset.csv", delimiter=",")

In [ ]:
base_repo = "NousResearch/Llama-2-7b-chat-hf"
adapter_name = "sentiment-lora-adapter"

In [ ]:
# 4-bit NF4 quantization keeps the frozen base weights small enough to fit on a single GPU;
# compute still happens in fp16 -- 4-bit is a storage format, not a compute one.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(base_repo, quantization_config=bnb_config, device_map="auto")
model.config.use_cache = False       # incompatible with gradient checkpointing during training
model.config.pretraining_tp = 1      # disable Llama's tensor-parallel pretraining path for fine-tuning

In [ ]:
# Llama-2 ships with no dedicated pad token, so eos is reused; padding on the right keeps
# new tokens after the real content rather than before it.
tokenizer = AutoTokenizer.from_pretrained(base_repo, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Rank-32 adapters injected into the attention layers; alpha < r here scales the adapter's
# contribution down relative to its capacity -- a conservative update.
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# SFTTrainer reads its training config from SFTConfig directly -- no separate
# TrainingArguments object is needed.
sft_config = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # simulates an effective batch size of 8
    warmup_steps=5,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_8bit",              # 8-bit optimizer state, cutting AdamW's memory cost
    weight_decay=0.01,
    lr_scheduler_type="linear",
    output_dir="outputs",
    report_to="none",
    dataset_text_field="train",      # CSV column that already holds the full formatted text
    packing=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=lora_config,
    args=sft_config,
)

trainer.train()

In [ ]:
trainer.model.save_pretrained(adapter_name)

In [ ]:
prompt = "It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement"

# Every forward pass here computes frozen_weight(x) + adapter(x) -- the adapter is not yet
# merged into the base model.
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]["generated_text"])

In [ ]:
# Free VRAM before reloading the base model fresh for merging -- both the quantized model
# and the trainer are no longer needed.
del model, trainer, pipe
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Reload the base model in fp16 (no quantization this time -- nothing left to train) and
# fold the LoRA delta directly into its weight matrices.
base_model = AutoModelForCausalLM.from_pretrained(
    base_repo, low_cpu_mem_usage=True, return_dict=True, torch_dtype=torch.float16, device_map="auto"
)
merged_model = PeftModel.from_pretrained(base_model, adapter_name).merge_and_unload()

In [ ]:
# Ordinary model + tokenizer pair, deployable anywhere a plain Llama-2 checkpoint would be --
# no PEFT dependency required at serving time.
merged_model.save_pretrained("sentiment-llama2-merged")
tokenizer.save_pretrained("sentiment-llama2-merged")

In [ ]:
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]["generated_text"])